# Análisis del Impacto de Variables Demográficas en la Predicción de Edad Cerebral

Este notebook analiza cómo las variables demográficas disponibles en la cohorte RedLaT
se relacionan con la brain age gap y si su inclusión como features mejora la predicción.

**Dos enfoques complementarios:**
1. **Post-hoc:** Correlación entre brain age gap (del modelo ya entrenado) y variables demográficas.
2. **Ablaciones:** Inclusión de variables demográficas como features adicionales en XGBoost.

In [1]:
import sys
from pathlib import Path

_cwd = Path.cwd()
if (_cwd / "src").exists():
    ROOT = _cwd
elif (_cwd.parent / "src").exists():
    ROOT = _cwd.parent
else:
    raise RuntimeError(f"Cannot locate project root from {_cwd}")

REPO = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print("Code root:", ROOT)
print("Repo root:", REPO)

Code root: /home/nico/Desktop/5to/TesisFinal/Thesis-comp-sci/Code
Repo root: /home/nico/Desktop/5to/TesisFinal/Thesis-comp-sci


In [2]:
import json
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from scipy import stats
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import KFold

from src.config import Paths, ExperimentConfig
from src.cohort import build_final_cohort_df
from src.data_io import read_full_metadata
from src.splits import load_splits
from src.xgb_train import clean_xy
from src.utils_ids import normalize_record_id

paths = Paths(
    excel_path=REPO / "Data" / "datos-redlat.xlsx",
    fc_folder=REPO / "Data" / "fc_mats",
    t1w_csv_path=REPO / "Data" / "Redlat_VGM_AAL_.csv",
    out_dir=REPO / "Outputs",
)
cfg = ExperimentConfig(seed=42)
np.random.seed(cfg.seed)

IMGDIR = REPO / "Latex" / "06_resultados_discusion" / "images"
IMGDIR.mkdir(parents=True, exist_ok=True)

mpl.rcParams.update({
    "figure.dpi": 150, "savefig.dpi": 300,
    "font.size": 11, "axes.titlesize": 12,
    "axes.spines.top": False, "axes.spines.right": False,
})

def save_fig(fname):
    plt.tight_layout()
    plt.savefig(IMGDIR / fname, bbox_inches="tight", pad_inches=0.05)
    plt.close()
    print("Saved:", IMGDIR / fname)

## 1. Cargar datos y artefactos del pipeline principal

In [3]:
cohort = build_final_cohort_df(
    excel_path=paths.excel_path,
    fc_folder=paths.fc_folder,
    t1w_csv_path=paths.t1w_csv_path,
    diagnoses_to_use=cfg.diagnoses_to_use,
)

splits = load_splits(paths.out_dir / "splits" / "splits_seed42_test0.1.json")
trainval_ids = splits["holdout"]["trainval_ids"]
test_ids = splits["holdout"]["test_ids"]

Z_trainval = np.load(paths.out_dir / "embeddings" / "mu_trainval.npy")
Z_test = np.load(paths.out_dir / "embeddings" / "mu_test.npy")

with open(paths.out_dir / "embeddings" / "mu_trainval.json") as f:
    emb_tv_meta = json.load(f)
with open(paths.out_dir / "embeddings" / "mu_test.json") as f:
    emb_te_meta = json.load(f)

emb_tv_ids = emb_tv_meta["ids"]
emb_te_ids = emb_te_meta["ids"]

print(f"Cohort: {len(cohort)}, Trainval: {len(trainval_ids)}, Test: {len(test_ids)}")
print(f"Embeddings trainval: {Z_trainval.shape}, test: {Z_test.shape}")

Cohort: 1245, Trainval: 1120, Test: 125
Embeddings trainval: (1120, 64), test: (125, 64)


In [4]:
xgb_params = json.load(open(paths.out_dir / "optuna" / "xgb_best_cv.json"))
if "best_params" in xgb_params:
    xgb_best = xgb_params["best_params"]
else:
    xgb_best = xgb_params

xgb_best["random_state"] = cfg.seed
xgb_best["eval_metric"] = "mae"
xgb_best["tree_method"] = "hist"
xgb_best["verbosity"] = 0
print("XGBoost best params:", {k: round(v, 4) if isinstance(v, float) else v for k, v in xgb_best.items()})

XGBoost best params: {'n_estimators': 2103, 'max_depth': 6, 'learning_rate': 0.011, 'subsample': 0.5021, 'colsample_bytree': 0.9506, 'reg_alpha': 0.0014, 'reg_lambda': 7.0842, 'min_child_weight': 1.1415, 'gamma': 0.6749, 'tree_method': 'hist', 'random_state': 42, 'eval_metric': 'mae', 'verbosity': 0}


## 2. Merge demográficos con cohorte

In [5]:
full_meta = read_full_metadata(paths.excel_path)

demo_cols = ["record_id", "site", "cog_ed", "mmse_total", "mri_scanner_brand"]
demo = full_meta[demo_cols].copy()

merged = cohort[["record_id", "age", "sex", "diagnosis"]].merge(
    demo, on="record_id", how="left"
)

print("Variables demográficas disponibles en la cohorte:")
for c in ["site", "cog_ed", "mmse_total", "mri_scanner_brand"]:
    n = merged[c].notna().sum()
    print(f"  {c}: {n}/{len(merged)} ({n/len(merged)*100:.1f}%)")

merged.head()

Variables demográficas disponibles en la cohorte:
  site: 941/1245 (75.6%)
  cog_ed: 933/1245 (74.9%)
  mmse_total: 923/1245 (74.1%)
  mri_scanner_brand: 921/1245 (74.0%)


,record_id,age,sex,diagnosis,site,cog_ed,mmse_total,mri_scanner_brand
0,AF025,75.0,Female,CN,Avila,16.0,29.0,2.0
1,AF036,74.0,Female,AD,Avila,11.0,22.0,2.0
2,AF037,71.0,Male,FTD,Avila,17.0,27.0,2.0
3,AF063,73.0,Female,CN,Avila,NaN,30.0,2.0
4,AF065,65.0,Male,CN,Avila,NaN,30.0,2.0


## 3. Descriptivos de variables demográficas

In [6]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4.5))

for i, diag in enumerate(["CN", "AD", "FTD"]):
    sub = merged[merged["diagnosis"] == diag]
    ed = sub["cog_ed"].dropna()
    axes[i].hist(ed, bins=15, edgecolor="white", alpha=0.8)
    axes[i].set_title(f"{diag} (n={len(ed)})")
    axes[i].set_xlabel("Años de educación")
    axes[i].set_ylabel("Sujetos")
    axes[i].axvline(ed.mean(), color="red", linestyle="--", linewidth=1.5,
                    label=f"μ={ed.mean():.1f}")
    axes[i].legend()

fig.suptitle("Distribución de años de educación por diagnóstico", y=1.02)
save_fig("demo_education_by_diag.png")

Saved: /home/nico/Desktop/5to/TesisFinal/Thesis-comp-sci/Latex/06_resultados_discusion/images/demo_education_by_diag.png


In [7]:
site_diag = pd.crosstab(merged["site"], merged["diagnosis"])
site_diag = site_diag[["CN", "AD", "FTD"]]
site_diag["Total"] = site_diag.sum(axis=1)
site_diag = site_diag.sort_values("Total", ascending=False)
print("Distribución por sitio y diagnóstico:")
print(site_diag)
print(f"\nSujetos sin sitio: {merged['site'].isna().sum()}")

Distribución por sitio y diagnóstico:
diagnosis    CN   AD  FTD  Total
site                            
Miller      177  102  121    400
Matallana    24   67   44    135
Slachevsky   45   33   15     93
Takada       71   11    7     89
Lopera        5   82    1     88
Behrens      26   24   10     60
Bruno        28   19    1     48
Resende       1    8   10     19
Avila         5    2    2      9

Sujetos sin sitio: 304


## 4. Brain Age Gap del modelo principal

In [8]:
t1_cols = [c for c in cohort.columns if c.startswith("t1_")]

def get_arrays(ids_list, Z_array, emb_ids):
    """Align cohort rows and embeddings to an ID list."""
    cohort_idx = {r: i for i, r in enumerate(cohort["record_id"].values)}
    emb_idx = {r: i for i, r in enumerate(emb_ids)}
    
    rows_c, rows_z = [], []
    valid_ids = []
    for rid in ids_list:
        if rid in cohort_idx and rid in emb_idx:
            rows_c.append(cohort_idx[rid])
            rows_z.append(emb_idx[rid])
            valid_ids.append(rid)
    
    c_sub = cohort.iloc[rows_c]
    T1 = c_sub[t1_cols].values.astype(np.float32)
    y = c_sub["age"].values.astype(np.float64)
    Z = Z_array[rows_z]
    X = np.concatenate([Z, T1], axis=1).astype(np.float32)
    X = np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0)
    return X, y, valid_ids, c_sub

X_trainval, y_trainval, ids_tv, df_tv = get_arrays(trainval_ids, Z_trainval, emb_tv_ids)
X_test, y_test, ids_te, df_te = get_arrays(test_ids, Z_test, emb_te_ids)

model = XGBRegressor(**xgb_best)
model.fit(X_trainval, y_trainval, verbose=False)

pred_test = model.predict(X_test)
gap_test = pred_test - y_test
mae_test = mean_absolute_error(y_test, pred_test)
r2_test = r2_score(y_test, pred_test)

print(f"Test MAE: {mae_test:.2f}, R²: {r2_test:.3f} (sanity check)")

test_results = pd.DataFrame({
    "record_id": ids_te,
    "age": y_test,
    "pred_age": pred_test,
    "gap": gap_test,
})
test_results = test_results.merge(merged, on="record_id", how="left", suffixes=("", "_dup"))
print(f"Test subjects with gap: {len(test_results)}")

Test MAE: 5.62, R²: 0.411 (sanity check)
Test subjects with gap: 125


In [9]:
kf = KFold(n_splits=5, shuffle=True, random_state=cfg.seed)
cv_gaps = []

for fold_i, (tr_idx, va_idx) in enumerate(kf.split(X_trainval)):
    m = XGBRegressor(**xgb_best)
    m.fit(X_trainval[tr_idx], y_trainval[tr_idx], verbose=False)
    pred_va = m.predict(X_trainval[va_idx])
    for j, idx in enumerate(va_idx):
        cv_gaps.append({
            "record_id": ids_tv[idx],
            "age": float(y_trainval[idx]),
            "pred_age": float(pred_va[j]),
            "gap": float(pred_va[j] - y_trainval[idx]),
            "fold": fold_i,
        })

cv_results = pd.DataFrame(cv_gaps)
cv_results = cv_results.merge(merged, on="record_id", how="left", suffixes=("", "_dup"))

all_results = pd.concat([
    cv_results.assign(split="trainval_cv"),
    test_results.assign(split="test"),
], ignore_index=True)

print(f"CV gaps: {len(cv_results)}, Test gaps: {len(test_results)}, Total: {len(all_results)}")
print(f"CV MAE: {mean_absolute_error(cv_results['age'], cv_results['pred_age']):.2f}")

CV gaps: 1120, Test gaps: 125, Total: 1245
CV MAE: 6.39


## 5. Análisis post-hoc: Gap vs. Educación

In [10]:
has_ed = all_results[all_results["cog_ed"].notna()].copy()
print(f"Sujetos con educación: {len(has_ed)} / {len(all_results)}")

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
colors = {"CN": "#4FC3F7", "AD": "#EF5350", "FTD": "#FFB74D"}

for i, diag in enumerate(["CN", "AD", "FTD"]):
    sub = has_ed[has_ed["diagnosis"] == diag]
    ed = sub["cog_ed"].values
    gap = sub["gap"].values
    
    r_pear, p_pear = stats.pearsonr(ed, gap)
    r_spear, p_spear = stats.spearmanr(ed, gap)
    
    axes[i].scatter(ed, gap, s=12, alpha=0.45, color=colors[diag])
    m, b = np.polyfit(ed, gap, 1)
    xs = np.linspace(ed.min(), ed.max(), 100)
    axes[i].plot(xs, m * xs + b, color="black", linewidth=1.5)
    axes[i].axhline(0, color="gray", linestyle="--", linewidth=0.8)
    axes[i].set_title(f"{diag} (n={len(sub)})")
    axes[i].set_xlabel("Años de educación")
    axes[i].set_ylabel("Brain age gap (años)")
    axes[i].annotate(
        f"r={r_pear:.3f} (p={p_pear:.3f})\nρ={r_spear:.3f} (p={p_spear:.3f})",
        xy=(0.05, 0.95), xycoords="axes fraction", va="top", fontsize=9,
        bbox=dict(boxstyle="round,pad=0.3", fc="white", alpha=0.8)
    )

fig.suptitle("Brain age gap vs. años de educación por diagnóstico", y=1.02)
save_fig("demo_gap_vs_education.png")

Sujetos con educación: 933 / 1245
Saved: /home/nico/Desktop/5to/TesisFinal/Thesis-comp-sci/Latex/06_resultados_discusion/images/demo_gap_vs_education.png


In [11]:
print("=== Correlación Gap vs. Educación ===")
for diag in ["CN", "AD", "FTD", "ALL"]:
    sub = has_ed if diag == "ALL" else has_ed[has_ed["diagnosis"] == diag]
    ed, gap = sub["cog_ed"].values, sub["gap"].values
    r_p, p_p = stats.pearsonr(ed, gap)
    r_s, p_s = stats.spearmanr(ed, gap)
    print(f"  {diag:4s} (n={len(sub):4d}): Pearson r={r_p:+.3f} (p={p_p:.4f}), "
          f"Spearman ρ={r_s:+.3f} (p={p_s:.4f}), mean gap={gap.mean():+.2f}")

=== Correlación Gap vs. Educación ===
  CN   (n= 374): Pearson r=-0.067 (p=0.1930), Spearman ρ=-0.091 (p=0.0790), mean gap=+0.73
  AD   (n= 348): Pearson r=+0.165 (p=0.0020), Spearman ρ=+0.157 (p=0.0032), mean gap=-1.12
  FTD  (n= 211): Pearson r=+0.002 (p=0.9805), Spearman ρ=-0.025 (p=0.7159), mean gap=+1.63
  ALL  (n= 933): Pearson r=+0.070 (p=0.0333), Spearman ρ=+0.038 (p=0.2506), mean gap=+0.25


## 6. Análisis post-hoc: Gap vs. Sitio

In [12]:
has_site = all_results[all_results["site"].notna()].copy()

site_stats = has_site.groupby("site")["gap"].agg(["count", "mean", "std"]).round(2)
site_stats = site_stats.sort_values("count", ascending=False)
site_stats.columns = ["N", "Mean Gap", "SD Gap"]
print("Gap por sitio (todos los diagnósticos):")
print(site_stats)

f_stat, p_anova = stats.f_oneway(
    *[grp["gap"].values for _, grp in has_site.groupby("site") if len(grp) >= 10]
)
print(f"\nANOVA (sitios con n≥10): F={f_stat:.2f}, p={p_anova:.4f}")

Gap por sitio (todos los diagnósticos):
              N  Mean Gap  SD Gap
site                             
Miller      400     -0.20    6.69
Matallana   135      0.92    6.21
Slachevsky   93      1.15    7.89
Takada       89      2.82    9.20
Lopera       88     -3.64    6.04
Behrens      60     -1.05    6.80
Bruno        48      4.30    8.80
Resende      19      2.03    7.93
Avila         9      0.71   12.46

ANOVA (sitios con n≥10): F=8.67, p=0.0000


In [13]:
top_sites = site_stats[site_stats["N"] >= 20].index.tolist()
plot_data = has_site[has_site["site"].isin(top_sites)]

fig, ax = plt.subplots(figsize=(10, 5))
site_order = plot_data.groupby("site")["gap"].mean().sort_values().index.tolist()
data_by_site = [plot_data[plot_data["site"] == s]["gap"].values for s in site_order]

bp = ax.boxplot(data_by_site, tick_labels=site_order, showfliers=False, patch_artist=True)
for patch in bp["boxes"]:
    patch.set_facecolor("#B3E5FC")
ax.axhline(0, color="gray", linestyle="--", linewidth=0.8)
ax.set_xlabel("Sitio de adquisición")
ax.set_ylabel("Brain age gap (años)")
ax.set_title(f"Brain age gap por sitio (ANOVA p={p_anova:.4f})")
plt.xticks(rotation=30, ha="right")
save_fig("demo_gap_by_site.png")

Saved: /home/nico/Desktop/5to/TesisFinal/Thesis-comp-sci/Latex/06_resultados_discusion/images/demo_gap_by_site.png


In [14]:
print("=== Gap por sitio — solo CN ===")
cn_site = has_site[has_site["diagnosis"] == "CN"]
cn_site_stats = cn_site.groupby("site")["gap"].agg(["count", "mean", "std"]).round(2)
cn_site_stats = cn_site_stats.sort_values("count", ascending=False)
cn_site_stats.columns = ["N", "Mean Gap", "SD Gap"]
print(cn_site_stats)

cn_groups = [grp["gap"].values for _, grp in cn_site.groupby("site") if len(grp) >= 5]
if len(cn_groups) >= 2:
    f_cn, p_cn = stats.f_oneway(*cn_groups)
    print(f"\nANOVA CN (sitios n≥5): F={f_cn:.2f}, p={p_cn:.4f}")

=== Gap por sitio — solo CN ===
              N  Mean Gap  SD Gap
site                             
Miller      177     -2.25    6.32
Takada       71      3.79    9.57
Slachevsky   45      3.15    8.63
Bruno        28      8.70    7.48
Behrens      26     -0.22    7.26
Matallana    24      1.47    5.77
Avila         5      4.43   15.89
Lopera        5     -1.67   10.07
Resende       1     14.87     NaN

ANOVA CN (sitios n≥5): F=10.82, p=0.0000


## 7. Análisis post-hoc: Gap vs. Scanner

In [15]:
has_scanner = all_results[all_results["mri_scanner_brand"].notna()].copy()
scanner_map = {0.0: "Otro", 1.0: "GE", 2.0: "Siemens", 4.0: "Philips"}
has_scanner["scanner"] = has_scanner["mri_scanner_brand"].map(scanner_map).fillna("Unknown")

scanner_stats = has_scanner.groupby("scanner")["gap"].agg(["count", "mean", "std"]).round(2)
scanner_stats.columns = ["N", "Mean Gap", "SD Gap"]
print("Gap por marca de scanner:")
print(scanner_stats.sort_values("N", ascending=False))

sc_groups = [grp["gap"].values for _, grp in has_scanner.groupby("scanner") if len(grp) >= 10]
if len(sc_groups) >= 2:
    f_sc, p_sc = stats.f_oneway(*sc_groups)
    print(f"\nANOVA scanner: F={f_sc:.2f}, p={p_sc:.4f}")

Gap por marca de scanner:
           N  Mean Gap  SD Gap
scanner                       
Siemens  581     -0.26    7.07
GE       289      0.95    7.52
Philips   47      4.33    8.89
Otro       4     -9.53    6.60

ANOVA scanner: F=9.91, p=0.0001


## 8. Regresión multivariable: Gap ~ demográficos

In [16]:
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import LabelEncoder

reg_data = all_results.dropna(subset=["cog_ed", "site"]).copy()
reg_data["sex_num"] = (reg_data["sex"] == "Male").astype(int)
reg_data["diag_ad"] = (reg_data["diagnosis"] == "AD").astype(int)
reg_data["diag_ftd"] = (reg_data["diagnosis"] == "FTD").astype(int)

site_dummies = pd.get_dummies(reg_data["site"], prefix="site", drop_first=True)
reg_data = pd.concat([reg_data, site_dummies], axis=1)

feature_sets = {
    "education": ["cog_ed"],
    "education + sex": ["cog_ed", "sex_num"],
    "education + sex + diagnosis": ["cog_ed", "sex_num", "diag_ad", "diag_ftd"],
    "education + sex + diagnosis + site": ["cog_ed", "sex_num", "diag_ad", "diag_ftd"] + list(site_dummies.columns),
}

print(f"Regression data: {len(reg_data)} subjects")
print("\n=== Regresión lineal: gap ~ variables demográficas ===")
for name, feats in feature_sets.items():
    X_reg = reg_data[feats].values.astype(float)
    y_gap = reg_data["gap"].values
    lr = LinearRegression().fit(X_reg, y_gap)
    r2 = lr.score(X_reg, y_gap)
    print(f"  {name:45s}: R²={r2:.4f}")
    if len(feats) <= 5:
        for fn, coef in zip(feats, lr.coef_):
            print(f"    {fn:25s}: β={coef:+.4f}")
        print(f"    {'intercept':25s}: {lr.intercept_:+.4f}")

Regression data: 933 subjects

=== Regresión lineal: gap ~ variables demográficas ===
  education                                    : R²=0.0049
    cog_ed                   : β=+0.1117
    intercept                : -1.3824
  education + sex                              : R²=0.0075
    cog_ed                   : β=+0.1063
    sex_num                  : β=-0.7774
    intercept                : -0.8186
  education + sex + diagnosis                  : R²=0.0262
    cog_ed                   : β=+0.0585
    sex_num                  : β=-0.6395
    diag_ad                  : β=-1.6842
    diag_ftd                 : β=+0.8723
    intercept                : +0.2224
  education + sex + diagnosis + site           : R²=0.0804


## 9. Análisis solo CN: Gap vs. Educación

In [17]:
cn_only = all_results[(all_results["diagnosis"] == "CN") & all_results["cog_ed"].notna()].copy()

ed_cn = cn_only["cog_ed"].values
gap_cn = cn_only["gap"].values
r_p, p_p = stats.pearsonr(ed_cn, gap_cn)
r_s, p_s = stats.spearmanr(ed_cn, gap_cn)

print(f"CN-only (n={len(cn_only)}):")
print(f"  Pearson: r={r_p:+.3f}, p={p_p:.4f}")
print(f"  Spearman: ρ={r_s:+.3f}, p={p_s:.4f}")
print(f"  Mean gap: {gap_cn.mean():+.2f} ± {gap_cn.std():.2f}")

ed_terciles = pd.qcut(cn_only["cog_ed"], 3, labels=["Low", "Mid", "High"])
cn_only = cn_only.assign(ed_group=ed_terciles.values)

print("\nGap por tercil de educación (solo CN):")
for g in ["Low", "Mid", "High"]:
    sub = cn_only[cn_only["ed_group"] == g]
    print(f"  {g}: n={len(sub)}, mean gap={sub['gap'].mean():+.2f} ± {sub['gap'].std():.2f}, "
          f"ed range=[{sub['cog_ed'].min():.0f}, {sub['cog_ed'].max():.0f}]")

groups_terc = [cn_only[cn_only["ed_group"] == g]["gap"].values for g in ["Low", "Mid", "High"]]
f_terc, p_terc = stats.f_oneway(*groups_terc)
print(f"\nANOVA terciles educación CN: F={f_terc:.2f}, p={p_terc:.4f}")

CN-only (n=374):
  Pearson: r=-0.067, p=0.1930
  Spearman: ρ=-0.091, p=0.0790
  Mean gap: +0.73 ± 8.13

Gap por tercil de educación (solo CN):
  Low: n=187, mean gap=+1.31 ± 8.44, ed range=[2, 16]
  Mid: n=112, mean gap=+0.16 ± 7.56, ed range=[17, 18]
  High: n=75, mean gap=+0.15 ± 8.20, ed range=[19, 27]

ANOVA terciles educación CN: F=0.94, p=0.3915


In [18]:
fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(ed_cn, gap_cn, s=18, alpha=0.5, color="#4FC3F7")
m, b = np.polyfit(ed_cn, gap_cn, 1)
xs = np.linspace(ed_cn.min(), ed_cn.max(), 100)
ax.plot(xs, m * xs + b, color="black", linewidth=1.5)
ax.axhline(0, color="gray", linestyle="--", linewidth=0.8)
ax.set_xlabel("Años de educación")
ax.set_ylabel("Brain age gap (años)")
ax.set_title(f"Controles sanos: gap vs. educación (n={len(cn_only)})")
ax.annotate(
    f"r={r_p:+.3f} (p={p_p:.3f})\nρ={r_s:+.3f} (p={p_s:.3f})",
    xy=(0.05, 0.95), xycoords="axes fraction", va="top", fontsize=10,
    bbox=dict(boxstyle="round,pad=0.3", fc="white", alpha=0.8)
)
save_fig("demo_cn_gap_vs_education.png")

Saved: /home/nico/Desktop/5to/TesisFinal/Thesis-comp-sci/Latex/06_resultados_discusion/images/demo_cn_gap_vs_education.png


## 10. Ablaciones: Educación y sitio como features adicionales

In [19]:
def build_demo_features(ids_list, Z_array, emb_ids, cohort_df, merged_df,
                        use_education=False, use_site=False):
    """Build feature matrix with optional demographic features."""
    t1_cols = [c for c in cohort_df.columns if c.startswith("t1_")]
    cohort_idx = {r: i for i, r in enumerate(cohort_df["record_id"].values)}
    emb_idx = {r: i for i, r in enumerate(emb_ids)}
    merged_idx = {r: i for i, r in enumerate(merged_df["record_id"].values)}
    
    valid_ids = []
    for rid in ids_list:
        if rid not in cohort_idx or rid not in emb_idx or rid not in merged_idx:
            continue
        row_m = merged_df.iloc[merged_idx[rid]]
        if use_education and pd.isna(row_m.get("cog_ed")):
            continue
        if use_site and pd.isna(row_m.get("site")):
            continue
        valid_ids.append(rid)
    
    c_rows = [cohort_idx[r] for r in valid_ids]
    e_rows = [emb_idx[r] for r in valid_ids]
    m_rows = [merged_idx[r] for r in valid_ids]
    
    c_sub = cohort_df.iloc[c_rows]
    Z = Z_array[e_rows]
    T1 = c_sub[t1_cols].values.astype(np.float32)
    y = c_sub["age"].values.astype(np.float64)
    
    parts = [Z, T1]
    feat_desc = f"Z({Z.shape[1]}) + T1({T1.shape[1]})"
    
    if use_education:
        ed = merged_df.iloc[m_rows]["cog_ed"].values.astype(np.float32).reshape(-1, 1)
        parts.append(ed)
        feat_desc += " + ed(1)"
    
    if use_site:
        site_vals = merged_df.iloc[m_rows]["site"]
        site_dum = pd.get_dummies(site_vals, prefix="site").values.astype(np.float32)
        parts.append(site_dum)
        feat_desc += f" + site({site_dum.shape[1]})"
    
    X = np.concatenate(parts, axis=1).astype(np.float32)
    X = np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0)
    return X, y, valid_ids, feat_desc

In [20]:
configs = [
    {"name": "VAE + T1w (baseline)",         "use_education": False, "use_site": False},
    {"name": "VAE + T1w + educación",         "use_education": True,  "use_site": False},
    {"name": "VAE + T1w + sitio",             "use_education": False, "use_site": True},
    {"name": "VAE + T1w + educación + sitio", "use_education": True,  "use_site": True},
]

ablation_results = []

for c in configs:
    X_tr, y_tr, ids_tr_c, desc_tr = build_demo_features(
        trainval_ids, Z_trainval, emb_tv_ids, cohort, merged,
        use_education=c["use_education"], use_site=c["use_site"]
    )
    X_te, y_te, ids_te_c, desc_te = build_demo_features(
        test_ids, Z_test, emb_te_ids, cohort, merged,
        use_education=c["use_education"], use_site=c["use_site"]
    )
    
    m = XGBRegressor(**xgb_best)
    m.fit(X_tr, y_tr, verbose=False)
    pred = m.predict(X_te)
    
    mae = mean_absolute_error(y_te, pred)
    r2 = r2_score(y_te, pred)
    pearson_r = float(np.corrcoef(y_te, pred)[0, 1])
    
    ablation_results.append({
        "config": c["name"],
        "n_train": len(X_tr), "n_test": len(X_te),
        "n_features": X_tr.shape[1],
        "MAE": mae, "R2": r2, "Pearson": pearson_r,
    })
    print(f"{c['name']:40s} | train={len(X_tr):4d}, test={len(X_te):3d}, "
          f"feats={X_tr.shape[1]:3d} | MAE={mae:.2f}, R²={r2:.3f}, r={pearson_r:.3f}")

ablation_df = pd.DataFrame(ablation_results)
print("\n" + ablation_df.to_string(index=False))

VAE + T1w (baseline)                     | train=1120, test=125, feats=180 | MAE=5.62, R²=0.411, r=0.644
VAE + T1w + educación                    | train= 842, test= 91, feats=181 | MAE=5.26, R²=0.504, r=0.715
VAE + T1w + sitio                        | train= 848, test= 93, feats=189 | MAE=5.51, R²=0.534, r=0.740
VAE + T1w + educación + sitio            | train= 842, test= 91, feats=190 | MAE=5.37, R²=0.483, r=0.699

                       config  n_train  n_test  n_features      MAE       R2  Pearson
         VAE + T1w (baseline)     1120     125         180 5.620665 0.411044 0.644342
        VAE + T1w + educación      842      91         181 5.262311 0.503930 0.715047
            VAE + T1w + sitio      848      93         189 5.512885 0.533641 0.740301
VAE + T1w + educación + sitio      842      91         190 5.368094 0.482723 0.698601


## 11. Resumen de resultados

In [21]:
print("=" * 70)
print("RESUMEN: Impacto de variables demográficas")
print("=" * 70)
print()
print("1. EDUCACIÓN vs. BRAIN AGE GAP")
for diag in ["CN", "AD", "FTD"]:
    sub = has_ed[has_ed["diagnosis"] == diag]
    r_p, p_p = stats.pearsonr(sub["cog_ed"].values, sub["gap"].values)
    print(f"   {diag}: r={r_p:+.3f}, p={p_p:.4f}, n={len(sub)}")
print()
print("2. SITIO: ANOVA sobre gap")
print(f"   F={f_stat:.2f}, p={p_anova:.4f}")
print()
print("3. ABLACIONES (educación/sitio como features)")
for _, row in ablation_df.iterrows():
    print(f"   {row['config']:40s}: MAE={row['MAE']:.2f}, R²={row['R2']:.3f} (n_test={row['n_test']})")
print()
print("4. CONCLUSIÓN")
print("   Los resultados deben interpretarse considerando que:")
print("   - ~25% de sujetos pierden dato de educación/sitio")
print("   - La reducción de N cambia el subset de test")
print("   - El modelo fue entrenado con todos los diagnósticos")

RESUMEN: Impacto de variables demográficas

1. EDUCACIÓN vs. BRAIN AGE GAP
   CN: r=-0.067, p=0.1930, n=374
   AD: r=+0.165, p=0.0020, n=348
   FTD: r=+0.002, p=0.9805, n=211

2. SITIO: ANOVA sobre gap
   F=8.67, p=0.0000

3. ABLACIONES (educación/sitio como features)
   VAE + T1w (baseline)                    : MAE=5.62, R²=0.411 (n_test=125)
   VAE + T1w + educación                   : MAE=5.26, R²=0.504 (n_test=91)
   VAE + T1w + sitio                       : MAE=5.51, R²=0.534 (n_test=93)
   VAE + T1w + educación + sitio           : MAE=5.37, R²=0.483 (n_test=91)

4. CONCLUSIÓN
   Los resultados deben interpretarse considerando que:
   - ~25% de sujetos pierden dato de educación/sitio
   - La reducción de N cambia el subset de test
   - El modelo fue entrenado con todos los diagnósticos


In [22]:
results_out = {
    "ablations": ablation_results,
    "correlations": {},
}
for diag in ["CN", "AD", "FTD", "ALL"]:
    sub = has_ed if diag == "ALL" else has_ed[has_ed["diagnosis"] == diag]
    r_p, p_p = stats.pearsonr(sub["cog_ed"].values, sub["gap"].values)
    r_s, p_s = stats.spearmanr(sub["cog_ed"].values, sub["gap"].values)
    results_out["correlations"][diag] = {
        "n": len(sub), "pearson_r": r_p, "pearson_p": p_p,
        "spearman_rho": r_s, "spearman_p": p_s,
        "mean_gap": float(sub["gap"].mean()),
    }

out_path = paths.out_dir / "experiments" / "demographics_analysis.json"
out_path.parent.mkdir(parents=True, exist_ok=True)
with open(out_path, "w") as f:
    json.dump(results_out, f, indent=2, default=float)
print("Saved:", out_path)

Saved: /home/nico/Desktop/5to/TesisFinal/Thesis-comp-sci/Outputs/experiments/demographics_analysis.json
